# Step 1: Data Understanding

**Dataset:** PaySim — a synthetic mobile-money transaction dataset generated to mimic real financial transaction logs, created for fraud-detection research (originally published by Lopez-Rojas, Elmir & Axelsson, 2016). Used on Kaggle as "Synthetic Financial Datasets For Fraud Detection".

File used: `data/raw/PS_20174392719_1491204439457_log.csv` (~493.5 MB).

**Goal of this notebook:** before writing any modeling code, understand what's actually in the data — size, types, missing values, duplicates, and how rare fraud actually is. This shapes every later decision (splitting strategy, imbalance handling, evaluation metrics).

In [ ]:
import sys
sys.path.append("..")

import pandas as pd

from src.data_utils import load_raw_data, find_raw_csv, COLUMN_DESCRIPTIONS

pd.set_option("display.max_columns", None)

## 1.1 Load the data

The full PaySim file has ~6.36 million rows (~493.5 MB as CSV) — manageable in memory on a laptop, so we load it in full rather than sampling.

In [ ]:
print("Reading from:", find_raw_csv())
df = load_raw_data()
df.shape

In [ ]:
df.head()

## 1.2 Column definitions

Understanding every column matters here because PaySim has a well-known **leakage trap**: the balance columns and `isFlaggedFraud` can make fraud trivially (and unrealistically) separable if used carelessly. See section 1.9 below.

In [ ]:
for col, desc in COLUMN_DESCRIPTIONS.items():
    print(f"{col:16s} -> {desc}")

## 1.3 Data types and basic structure

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## 1.4 Missing values

Fraud models are sensitive to how missingness is handled — silently dropping or filling rows can bias the fraud rate. Let's check if PaySim even has any.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})

## 1.5 Duplicate rows

In [ ]:
n_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {n_dupes} ({n_dupes / len(df) * 100:.4f}%)")

## 1.6 Categorical columns: unique values

`type` has a small, fixed set of categories. `nameOrig`/`nameDest` are near-unique customer/merchant IDs — useful for behavioral aggregation later, not usable directly as categorical features.

In [ ]:
df["type"].value_counts()

In [ ]:
print(f"nameOrig unique: {df['nameOrig'].nunique()} out of {len(df)} rows")
print(f"nameDest unique: {df['nameDest'].nunique()} out of {len(df)} rows")

## 1.7 Fraud vs. legitimate transaction distribution

This is the single most important number in the whole project. Fraud detection is a classic **severe class imbalance** problem — a model that always predicts "legit" would score ~99.87% accuracy while being completely useless. This is why later steps explicitly avoid accuracy as the headline metric.

In [ ]:
fraud_counts = df["isFraud"].value_counts()
fraud_pct = df["isFraud"].value_counts(normalize=True) * 100

summary = pd.DataFrame({"count": fraud_counts, "pct": fraud_pct})
summary.index = ["Legitimate (0)", "Fraud (1)"]
summary

In [ ]:
ratio = fraud_counts[0] / fraud_counts[1]
print(f"For every 1 fraud transaction, there are ~{ratio:.2f} legitimate ones.")

## 1.8 Which transaction types does fraud occur in?

Worth checking early, since it directly informs feature engineering and error analysis later.

In [ ]:
df[df["isFraud"] == 1]["type"].value_counts()

## 1.9 `isFlaggedFraud` and leakage concerns

PaySim ships with its own naive rule-based flag (`isFlaggedFraud`, documented as triggering on large TRANSFERs). We check how well it actually covers real fraud, and flag it as **off-limits as a model feature** — it's a rule output, not an input signal, so training on it would be leaking a partial answer into the model.

The balance columns (`oldbalanceOrg`, `newbalanceOrig`, `oldbalanceDest`, `newbalanceDest`) are legitimate pre-transaction signals in principle, but PaySim's simulation quirks (e.g. destination balances often sitting at exactly 0 before/after a transaction) can make fraud unrealistically easy to separate if used naively as raw balances rather than as engineered *differences*/*ratios*. This will be handled carefully in the feature engineering step — not now.

In [ ]:
pd.crosstab(df["isFraud"], df["isFlaggedFraud"], margins=True)

## 1.10 Notes / takeaways (from the actual run on the full file)

- **Shape:** 6,362,620 rows x 11 columns.
- **Columns:** `step`, `type`, `amount`, `nameOrig`, `oldbalanceOrg`, `newbalanceOrig`, `nameDest`, `oldbalanceDest`, `newbalanceDest`, `isFraud`, `isFlaggedFraud`.
- **Dtypes:** `step`/`isFraud`/`isFlaggedFraud` are int64; `amount` and the four balance columns are float64; `type`, `nameOrig`, `nameDest` are object (string).
- **Missing values:** none in any column (0 across the board).
- **Duplicate rows:** 0 exact duplicates.
- **Transaction types:** CASH_OUT (2,237,500), PAYMENT (2,151,495), CASH_IN (1,399,284), TRANSFER (532,909), DEBIT (41,432).
- **ID cardinality:** 6,353,307 unique `nameOrig` values out of 6,362,620 rows (almost every sender is distinct — most customers transact once or a handful of times in this simulation); 2,722,362 unique `nameDest` values (recipients, especially merchants, repeat more).
- **Fraud distribution:** 8,213 fraud / 6,354,407 legitimate -> fraud is **0.1291%** of transactions, legitimate is 99.8709%. Ratio is roughly **774 legitimate transactions for every 1 fraud** transaction. This is a severe imbalance -- accuracy is not a usable metric on its own.
- **Fraud only occurs in two transaction types:** TRANSFER (4,097 fraud cases) and CASH_OUT (4,116 fraud cases). PAYMENT, CASH_IN, and DEBIT have zero fraud in this dataset. This is a strong, genuine signal we'll lean on in feature engineering -- but also a reason to be cautious: a model could learn to just gate on `type`, so recall within TRANSFER/CASH_OUT specifically will matter more than aggregate recall.
- **`isFlaggedFraud` is almost useless as a detector:** only 16 transactions are flagged in the entire dataset, and all 16 are true fraud cases (precision 100% on those 16) -- but that means it catches only 16 out of 8,213 actual frauds (**recall ~0.19%**). Confirms we need a real ML model, not this simple threshold rule, and confirms `isFlaggedFraud` must **not** be used as an input feature.